<a href="https://colab.research.google.com/github/Suprov007/LLM_Based_OCR_-Analysis/blob/main/Qwen/Qwen2_5OCR__final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -U qwen-vl-utils
%pip install -U git+https://github.com/huggingface/transformers accelerate

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-v55wtxvl
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-v55wtxvl
  Resolved https://github.com/huggingface/transformers to commit 57f9936a2619d2f2d4af89bde34d5eb611c2b728
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
!pip install jiwer

In [ ]:
import sys
import torch

print("Python path:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Torch CUDA:", torch.version.cuda)

Python path: /usr/bin/python3
Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Torch CUDA: 12.8


In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch
from PIL import Image
import matplotlib.pyplot as plt
import requests
import pandas as pd
import re
from jiwer import wer, cer

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
import torch

model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

processor = AutoProcessor.from_pretrained(model_name)

print("Model loaded")

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

## Image Download

In [ ]:
import os
import platform

pages = [8, 114, 468, 316, 753]

"""
pages = [8, 9, 12, 38, 53, 78, 94, 105, 114, 115,
       146, 210, 314, 316, 457, 468, 490, 619, 615, 753]
"""

base = "https://nlp.fi.muni.cz/projekty/ahisto/book_by_id/494/"

os.makedirs("images", exist_ok=True)
os_type = platform.system()
print("OS: ", os_type)

for p in pages:
    path = f"images/{p}.jpg"

    if not os.path.exists(path):
        if os_type == "Windows":
            !curl -o {path} {base}{p}.jpg
        else:
            !wget -O {path} {base}{p}.jpg
    else:
        print(f"Already exists: {path}")

# Experiment 1 - Clean Baseline
Original image to OCR, Without preprocessing

In [ ]:
import os

output_dir = "outputs/Experiment_1"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
prompt_text_0 = "Extract all text from this image exactly as written. Preserve line breaks."

In [ ]:
prompt_text_1 ="""You are an expert in OCR transcription of historical German printed texts (15th–19th century).

Your goal is MAXIMUM CHARACTER ACCURACY for Fraktur print.

STRICT RULES:
- Do NOT guess unclear text.
- If uncertain, mark characters with [?].
- If unreadable, write [illegible].

Fraktur-specific constraints:
- Long s (ſ) must be transcribed as "ſ" (NOT f).
- Distinguish:
  - ſ vs f
  - rn vs m
  - c vs e
  - k vs h
- Preserve ligatures if visible (e.g., "ch", "tz", "ck").
- Keep abbreviations EXACT (e.g., "lb.", "sh.", "qr.", "u.", "etc.").

Language constraints:
- Do NOT modernize spelling.
- Do NOT correct grammar.
- Do NOT expand abbreviations.

ADDITIONAL INSTRUCTIONS:
- Process EACH word letter-by-letter.
- Before finalizing a word, verify each character visually.
- If two interpretations are possible, choose the one that best matches Fraktur shapes, NOT meaning.
- Do not use context to "fix" words unless visually certain.

Output:
Plain text transcription with preserved line breaks."""

In [ ]:
prompt_text_2 = """You are an expert in OCR transcription of historical German printed texts (15th–19th century).

STRICT RULES:
- Do NOT guess unclear text.
- If uncertain, mark characters with [?].
- If unreadable, write [illegible].

Fraktur-specific constraints:
- Long s (ſ) must be transcribed as "ſ" (NOT f).
- Distinguish:
  - ſ vs f
  - rn vs m
  - c vs e
  - k vs h
- Preserve ligatures if visible (e.g., "ch", "tz", "ck").
- Keep abbreviations EXACT (e.g., "lb.", "sh.", "qr.", "u.", "etc.").

Language constraints:
- Do NOT modernize spelling.
- Do NOT correct grammar.
- Do NOT expand abbreviations.

TASK:
For each line:
1. Transcribe text
2. Assign confidence score (0.0–1.0)

Confidence guidelines:
- 1.0 = fully clear
- 0.7–0.9 = minor ambiguity
- <0.7 = uncertain characters present

Format:
[text] || confidence=0.xx"""

In [ ]:
def run_ocr_one_image(image_path, prompt_text):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {
                    "type": "text",
                    "text": prompt_text
                },
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )

    inputs = inputs.to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=1500,
        do_sample=False
    )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return output_text

In [ ]:
for p in pages:
    print(f"\n==============================")
    print(f"Page {p}")
    print(f"==============================")

    image_path = f"images/{p}.jpg"
    output_path = os.path.join(output_dir, f"page_{p}.txt")

    output_text = run_ocr_one_image(image_path, prompt_text_0)

    #print
    print(output_text)

    # save in folder
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(output_text)

    print(f"Saved: {output_path}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# source (Colab)
src = "outputs/Experiment_1"

# destination (Google Drive)
dst = "/content/drive/MyDrive/German_Historical_Book/Qwen2.5/outputs/Experiment_1"

# create destination folder if not exists
os.makedirs(dst, exist_ok=True)


for file_name in os.listdir(src):
    shutil.copy(
        os.path.join(src, file_name),
        os.path.join(dst, file_name)
    )

print("All files copied to Google Drive.")

### Evaluation Experiment 1

In [ ]:
def normalize_text(text):
    text = text.lower()
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [ ]:
results = []

gold_base = "https://nlp.fi.muni.cz/projekty/ahisto/ocr-output/494/"

for p in pages:
    print(f"Evaluating page {p}...")

    # 1. Load gold text from URL
    gold_url = f"{gold_base}{p}.txt"
    gold_text = requests.get(gold_url).text

    # 2. Load your OCR output
    pred_path = os.path.join(output_dir, f"page_{p}.txt")
    with open(pred_path, "r", encoding="utf-8") as f:
        pred_text = f.read()

    # 3. Normalize both
    gold_norm = normalize_text(gold_text)
    pred_norm = normalize_text(pred_text)

    # 4. Compute metrics
    wer_score = wer(gold_norm, pred_norm)
    cer_score = cer(gold_norm, pred_norm)

    results.append({
        "page": p,
        "WER": wer_score,
        "CER": cer_score
    })

In [ ]:
df = pd.DataFrame(results)
df

In [ ]:
mean_wer = df["WER"].mean()
mean_cer = df["CER"].mean()

print("=== Experiment 1 Results ===")
print(f"Mean WER: {mean_wer:.4f}")
print(f"Mean CER: {mean_cer:.4f}")

# Experiment 2 - Noise
- Without Preprocessing
- Adding Noise

In [ ]:
import os

output_dir2 = "outputs/Experiment_2"
os.makedirs(output_dir2, exist_ok=True)

### Salt and Pepper Function

Salt-and-pepper noise was implemented based on standard image processing techniques as described in (https://www.geeksforgeeks.org/python/add-a-salt-and-pepper-noise-to-an-image-with-python/)

In [ ]:
import numpy as np
from PIL import Image


# amount = how many pixels change
# amount = 0.2 -- 20% pixel changed, amount = 0.8 -- 80% pixel changed
# salt_vs_pepper = how many are white
# salt_vs_pepper = 0.2 -- 20% white, salt_vs_pepper = 0.8 -- 80% white


def add_salt_pepper(image, amount=0.005, salt_vs_pepper=0.5):

    img = np.array(image).copy()
    h, w = img.shape[:2]

    num_pixels = int(amount * h * w)
    num_salt = int(num_pixels * salt_vs_pepper)
    num_pepper = num_pixels - num_salt

    #salt -white
    ys = np.random.randint(0, h, num_salt)
    xs = np.random.randint(0, w, num_salt)
    img[ys, xs] = 255

    #pepper -black
    ys = np.random.randint(0, h, num_pepper)
    xs = np.random.randint(0, w, num_pepper)
    img[ys, xs] = 0

    return Image.fromarray(img)

### Image_Distortion
The image distortion functions are adapted from the Image_Distortion repository(https://github.com/namks/Image_Distortion/tree/main), specifically the zoom and resolution degradation methods.

In [ ]:
!git clone https://github.com/namks/Image_Distortion.git


In [ ]:
import sys
sys.path.append("/content/Image_Distortion")

In [ ]:

#from distort import zoom_img, lower_res_img
from PIL import Image

def zoom_img(image, zoom_scale=1.15):
    w, h = image.size
    x = w / 2
    y = h / 2

    cropped = image.crop((
        x - w / (2 * zoom_scale),
        y - h / (2 * zoom_scale),
        x + w / (2 * zoom_scale),
        y + h / (2 * zoom_scale)
    ))

    return cropped.resize((w, h), Image.BICUBIC)

def lower_res_img(image, scale=0.7):
    w, h = image.size
    low_res = image.resize((int(w * scale), int(h * scale)), Image.BICUBIC)
    return low_res.resize((w, h), Image.BICUBIC)

def add_image_distortion(image, zoom_scale=1.15, resolution_scale=0.7):
    image = zoom_img(image, zoom_scale=zoom_scale)
    image = lower_res_img(image, scale=resolution_scale)
    return image

### Noisy
Additional blurring effects were applied to simulate realistic degradation, inspired by data augmentation techniques such as Noisify
(https://github.com/dstl/Noisify)

In [ ]:
import cv2
import numpy as np
from PIL import Image

def add_blur(image, kernel_size=3):
    img = np.array(image)
    blurred = cv2.GaussianBlur(img, (kernel_size, kernel_size), 0)
    return Image.fromarray(blurred)

### Testing one image with Noise





#### Test Salt and Pepper

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

noise_image_test = "/content/images/8.jpg"

img = Image.open(noise_image_test).convert("RGB")

plt.figure(figsize=(10,12))
plt.imshow(img)
plt.title("Original", fontsize=16)
plt.axis("off")
plt.show()

In [ ]:
noisy_img = add_salt_pepper(img, amount=0.02,salt_vs_pepper=0.2)
plt.figure(figsize=(10,12))
plt.imshow(noisy_img)
plt.title("Noisy", fontsize=16)
plt.axis("off")
plt.show()

#### Test Image Distortion and BLurr

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

noise_image_test2 = "/content/images/753.jpg"
img1 = Image.open(noise_image_test2).convert("RGB")

# FIXED zoom_scale
distorted_img = add_image_distortion(
    img1,
    zoom_scale=0.9,
    resolution_scale=0.8
)

# FIXED variable name
blurred_img = add_blur(img1, kernel_size=5)

# Original
plt.figure(figsize=(10,14))
plt.imshow(img1)
plt.title("Original", fontsize=18)
plt.axis("off")
plt.show()

# Distorted
plt.figure(figsize=(10,14))
plt.imshow(distorted_img)
plt.title("Distorted", fontsize=18)
plt.axis("off")
plt.show()

# Blur (optional to display)
plt.figure(figsize=(10,14))
plt.imshow(blurred_img)
plt.title("Blurred", fontsize=18)
plt.axis("off")
plt.show()

### Combining Three Functions
- add salt_pepper
- add image_distortion
- add blurr

In [ ]:
def apply_all_corruptions(image,sp_amount=0.005,sp_ratio=0.5,zoom_scale=1.15,resolution_scale=0.7,
                          blur_kernel=3):
    image = add_salt_pepper(
        image,
        amount=sp_amount,
        salt_vs_pepper=sp_ratio
    )

    image = add_image_distortion(
        image,
        zoom_scale=zoom_scale,
        resolution_scale=resolution_scale
    )

    image = add_blur(
        image,
        kernel_size=blur_kernel
    )

    return image

In [ ]:
noise_image_test = "/content/images/316.jpg"

img = Image.open(noise_image_test).convert("RGB")
combined_img = apply_all_corruptions(
    img,
    sp_amount=0.05,
    sp_ratio=0.3,
    zoom_scale=0.9,
    resolution_scale = 0.85,
    blur_kernel=5
)
# Original (big)
plt.figure(figsize=(10,14), dpi=150)
plt.imshow(img)
plt.title("Original", fontsize=18)
plt.axis("off")
plt.show()

# Combined (big)
plt.figure(figsize=(10,14), dpi=150)
plt.imshow(combined_img)
plt.title("Salt + Distortion + Blur", fontsize=18)
plt.axis("off")
plt.show()

### Apply corruption for all images

In [ ]:
import os
os.makedirs("corrupted_images", exist_ok=True)

In [ ]:
from PIL import Image

for p in pages:
    img = Image.open(f"images/{p}.jpg").convert("RGB")

    combined_img = apply_all_corruptions(
        img,
        sp_amount=0.05,
        sp_ratio=0.3,
        zoom_scale=0.9,
        resolution_scale=0.85,
        blur_kernel=5
    )

    combined_img.save(f"corrupted_images/{p}.jpg")

print("Done saving all corrupted images.")

In [ ]:
for p in pages:
    print(f"\n==============================")
    print(f"Page {p}")
    print(f"==============================")

    image_path = f"corrupted_images/{p}.jpg"
    output_path = os.path.join(output_dir2, f"page_{p}.txt")

    output_text = run_ocr_one_image(image_path, prompt_text_0)

    #print
    print(output_text)

    # save in folder
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(output_text)

    print(f"Saved: {output_path}")

### Storing files to Google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

import shutil
import os

# source (Colab)
src = "outputs/Experiment_2"

# destination (Google Drive)
dst = "/content/drive/MyDrive/German_Historical_Book/Qwen2.5/outputs/Experiment_2"

# create destination folder if not exists
os.makedirs(dst, exist_ok=True)


for file_name in os.listdir(src):
    shutil.copy(
        os.path.join(src, file_name),
        os.path.join(dst, file_name)
    )

print("All files copied to Google Drive.")

### Evaluation - Experiment 2

In [ ]:
def normalize_text(text):
    text = text.lower()
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [ ]:
results_exp2 = []

gold_base = "https://nlp.fi.muni.cz/projekty/ahisto/ocr-output/494/"

for p in pages:
    print(f"Evaluating Experiment 2 - page {p}...")

    # 1. Load gold text from URL
    gold_url = f"{gold_base}{p}.txt"
    gold_text = requests.get(gold_url).text

    # 2. Load your OCR output from Experiment 2
    pred_path = os.path.join(output_dir2, f"page_{p}.txt")
    with open(pred_path, "r", encoding="utf-8") as f:
        pred_text = f.read()

    # 3. Normalize both
    gold_norm = normalize_text(gold_text)
    pred_norm = normalize_text(pred_text)

    # 4. Compute metrics
    wer_score = wer(gold_norm, pred_norm)
    cer_score = cer(gold_norm, pred_norm)

    results_exp2.append({
        "page": p,
        "WER": wer_score,
        "CER": cer_score
    })

In [ ]:
df_exp2 = pd.DataFrame(results_exp2)
df_exp2

In [ ]:
mean_wer_exp2 = df_exp2["WER"].mean()
mean_cer_exp2 = df_exp2["CER"].mean()

print("=== Experiment 2 Results ===")
print(f"Mean WER: {mean_wer_exp2:.4f}")
print(f"Mean CER: {mean_cer_exp2:.4f}")

# Experiment 3 - With Preprocessing


In [ ]:
import cv2
import numpy as np
from PIL import Image

def preprocess_image(image, use_threshold=False):
    """
    Preprocessing for historical book page scans:
    - converts to single-channel document image
    - light denoising
    - contrast normalization
    - optional thresholding
    """
    img = np.array(image)

    #if image is RGB, convert to grayscale.
    #if it is already grayscale-like, this is still safe.
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray = img.copy()

    #light denoising: good for tiny scan specks
    denoised = cv2.medianBlur(gray, 3)

    #improve contrast
    normalized = cv2.equalizeHist(denoised)

    #optional thresholding
    if use_threshold:
        processed = cv2.threshold(
            normalized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )[1]
    else:
        processed = normalized

    return Image.fromarray(processed)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

test_path = "images/105.jpg"
img = Image.open(test_path).convert("RGB")

preprocessed_img = preprocess_image_book(img, use_threshold=False)

plt.figure(figsize=(10,14), dpi=150)
plt.imshow(img)
plt.title("Original", fontsize=18)
plt.axis("off")
plt.show()

plt.figure(figsize=(10,14), dpi=150)
plt.imshow(preprocessed_img, cmap="gray")
plt.title("Preprocessed", fontsize=18)
plt.axis("off")
plt.show()

In [ ]:
import os
from PIL import Image

os.makedirs("preprocessed_images", exist_ok=True)

for p in pages:
    img = Image.open(f"images/{p}.jpg").convert("RGB")

    preprocessed_img = preprocess_image(img, use_threshold=False)

    preprocessed_img.save(f"preprocessed_images/{p}.jpg")

print("Done saving all preprocessed images.")

In [ ]:
import os

output_dir3 = "outputs/Experiment_3"
os.makedirs(output_dir3, exist_ok=True)

for p in pages:
    print(f"\n==============================")
    print(f"Page {p}")
    print(f"==============================")

    image_path = f"preprocessed_images/{p}.jpg"
    output_path = os.path.join(output_dir3, f"page_{p}.txt")

    output_text = run_ocr_one_image(image_path, prompt_text_0)

    print(output_text)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(output_text)

    print(f"Saved: {output_path}")

### Evaluation Experiment 3

In [ ]:
def normalize_text(text):
    text = text.lower()
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [ ]:
results_exp3 = []

gold_base = "https://nlp.fi.muni.cz/projekty/ahisto/ocr-output/494/"

for p in pages:
    print(f"Evaluating Experiment 3 - page {p}...")

    # 1. Load gold text from URL
    gold_url = f"{gold_base}{p}.txt"
    gold_text = requests.get(gold_url).text

    # 2. Load your OCR output from Experiment 3
    pred_path = os.path.join(output_dir3, f"page_{p}.txt")
    with open(pred_path, "r", encoding="utf-8") as f:
        pred_text = f.read()

    # 3. Normalize both
    gold_norm = normalize_text(gold_text)
    pred_norm = normalize_text(pred_text)

    # 4. Compute metrics
    wer_score = wer(gold_norm, pred_norm)
    cer_score = cer(gold_norm, pred_norm)

    results_exp3.append({
        "page": p,
        "WER": wer_score,
        "CER": cer_score
    })

In [ ]:
df_exp3 = pd.DataFrame(results_exp3)
df_exp3

In [ ]:
mean_wer_exp3 = df_exp3["WER"].mean()
mean_cer_exp3 = df_exp3["CER"].mean()

print("=== Experiment 3 Results ===")
print(f"Mean WER: {mean_wer_exp3:.4f}")
print(f"Mean CER: {mean_cer_exp3:.4f}")

# Evaluation Summary
- Experiment 1
- Experiment 2
- Experiment 3

In [ ]:
summary_df = pd.DataFrame([
    {
        "Experiment": "Experiment 1",
        "Description": "Original images, no preprocessing",
        "Mean WER": mean_wer,
        "Mean CER": mean_cer
    },
    {
        "Experiment": "Experiment 2",
        "Description": "Corrupted images, no preprocessing",
        "Mean WER": mean_wer_exp2,
        "Mean CER": mean_cer_exp2
    },
    {
        "Experiment": "Experiment 3",
        "Description": "Original images, with preprocessing",
        "Mean WER": mean_wer_exp3,
        "Mean CER": mean_cer_exp3
    }
])

summary_df